### Data Exploration to afinity datasets

Datasets: 
1. ChEMBL 
2. PDSP
3. BindingDB

##### **0° INTRO**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR /  Path("data/raw")

DATA_CHEMBL = DATA_DIR / "chembl" / "chembl214_ki_ic50.parquet"

BINDINGDB_DIR = DATA_DIR / "BINDINGDB" / "BindingDB_BindingDB_Articles.tsv"
PDSP_DIR = DATA_DIR / "PDSP" / "BindingDB_PDSPKi.tsv"

#### **1° Afinity Datasets**

##### **Data by ChEMBL**

In [3]:
# TODO: Explorar los conjuntos de datos de afinidad y actividad biológica para identificar patrones, correlaciones y posibles relaciones entre las características moleculares y la afinidad hacia el receptor 5HT1A.

data_chembl = pd.read_parquet(DATA_CHEMBL)
data_summary = pd.DataFrame({
    'Columns': data_chembl.columns,
    'Data types': data_chembl.dtypes,
    'Missing values': data_chembl.isnull().sum(),
    'null%': data_chembl.isnull().mean() * 100,
    # 'unique values': data_chembl.nunique()
})

pd.set_option('display.max_rows', None)
display(data_summary)


,Columns,Data types,Missing values,null%
action_type,action_type,object,7132,91.283758
activity_comment,activity_comment,str,6760,86.522463
activity_id,activity_id,int64,0,0.000000
activity_properties,activity_properties,object,0,0.000000
assay_chembl_id,assay_chembl_id,str,0,0.000000
assay_description,assay_description,str,0,0.000000
assay_type,assay_type,str,0,0.000000
assay_variant_accession,assay_variant_accession,object,7813,100.000000
assay_variant_mutation,assay_variant_mutation,object,7813,100.000000
bao_endpoint,bao_endpoint,str,0,0.000000


In [20]:

# ? What is the observational unit of the data?

print(data_chembl[['assay_type', 'type', 'units', 'assay_description']])

     assay_type       type        units  \
0             B         Ki           nM   
1             B         Ki           nM   
2             B         Ki           nM   
3             B         Ki           nM   
4             B         Ki           nM   
5             B         Ki           nM   
6             B         Ki           nM   
7             B         Ki           nM   
8             B     Log Ki          NaN   
9             B     Log Ki          NaN   
10            B         Ki           nM   
11            B         Ki           nM   
12            B         Ki           nM   
13            B         Ki           nM   
14            B     Log Ki          NaN   
15            B         Ki           nM   
16            B         Ki           nM   
17            B         Ki           nM   
18            B      pIC50          NaN   
19            B      pIC50          NaN   
20            B      pIC50          NaN   
21            B      pIC50          NaN   
22         

In [5]:

# ? Ki or IC50? Which one is more relevant for our analysis?

print(data_chembl["type"].value_counts().T)
print(data_chembl["standard_type"].value_counts().T)
print(data_chembl["standard_relation"].value_counts().T)
print(data_chembl["standard_units"].value_counts().T)


type
Ki           5770
pKi           950
IC50          654
Log Ki        248
pIC50          86
-Log IC50      83
Log IC50       19
Ki(app)         3
Name: count, dtype: int64
standard_type
Ki      6971
IC50     842
Name: count, dtype: int64
standard_relation
=     6933
>      568
<       68
>=       1
Name: count, dtype: int64
standard_units
nM    7584
%       13
Name: count, dtype: int64


In [6]:

# ? Which molecules have unique SMILES representations? Are there any duplicates or inconsistencies in the SMILES strings?

print(data_chembl["canonical_smiles"].nunique())
print(data_chembl["canonical_smiles"].duplicated().sum())
print(data_chembl["pchembl_value"].describe())

6094
1718
count     6879
unique     575
top       8.40
freq        62
Name: pchembl_value, dtype: object


In [7]:
# TODO: Create groups based on unique SMILES representations and visualize the distribution of pChEMBL values for each group. This will help identify any patterns or trends in the data that may be relevant to our analysis of the 5HT1A receptor.


molecule_groups = data_chembl.groupby(['canonical_smiles', 'molecule_chembl_id']).size()
molecule_groups = molecule_groups.reset_index(name='count')
molecules_with_duplicates = molecule_groups[molecule_groups['count'] > 1]
print(f"Molecules with duplicate SMILES representations: {molecules_with_duplicates.shape[0]}")

Molecules with duplicate SMILES representations: 1139


In [8]:
def molecule_unique_smiles(molecules):
    df = molecules[('molecule_chembl_id', 'count')]


##### **Data by BidingDB**

In [9]:
data_bindb = pd.read_csv(BINDINGDB_DIR, sep='\t')
data_bindb.info()

C:\Users\alexl\AppData\Local\Temp\ipykernel_12564\1711188053.py:1: DtypeWarning: Columns (0: Ki (nM), 1: IC50 (nM), 2: Kd (nM), 3: EC50 (nM), 4: koff (s-1), 5: Temp (C), 6: Article DOI, 7: ChEMBL ID of Ligand, 8: DrugBank ID of Ligand, 9: KEGG ID of Ligand, 10: ZINC ID of Ligand, 11: UniProt (SwissProt) Secondary ID(s) of Target Chain 1, 12: UniProt (TrEMBL) Submitted Name of Target Chain 1, 13: UniProt (TrEMBL) Entry Name of Target Chain 1, 14: UniProt (TrEMBL) Primary ID of Target Chain 1, 15: UniProt (TrEMBL) Secondary ID(s) of Target Chain 1, 16: BindingDB Target Chain Sequence 2, 17: PDB ID(s) of Target Chain 2, 18: UniProt (SwissProt) Recommended Name of Target Chain 2, 19: UniProt (SwissProt) Entry Name of Target Chain 2, 20: UniProt (SwissProt) Primary ID of Target Chain 2, 21: UniProt (SwissProt) Secondary ID(s) of Target Chain 2, 22: UniProt (TrEMBL) Submitted Name of Target Chain 2, 23: UniProt (TrEMBL) Entry Name of Target Chain 2, 24: UniProt (TrEMBL) Primary ID of Target 

<class 'pandas.DataFrame'>
RangeIndex: 93712 entries, 0 to 93711
Columns: 640 entries, BindingDB Reactant_set_id to UniProt (TrEMBL) Alternative ID(s) of Target Chain 50
dtypes: float64(534), int64(3), object(5), str(98)
memory usage: 708.2+ MB


In [10]:
data_summary = pd.DataFrame({
    'Columns': data_bindb.columns,
    'Data types': data_bindb.dtypes,
    'Missing values': data_bindb.isnull().sum(),
    'null%': data_bindb.isnull().mean() * 100,
    'unique values': data_bindb.nunique()
})

pd.set_option('display.max_rows', None)
display(data_summary)

,Columns,Data types,Missing values,null%,unique values
BindingDB Reactant_set_id,BindingDB Reactant_set_id,int64,0,0.000000,93522
Ligand SMILES,Ligand SMILES,str,57,0.060825,47940
Ligand InChI,Ligand InChI,str,51,0.054422,46304
Ligand InChI Key,Ligand InChI Key,str,51,0.054422,46304
BindingDB MonomerID,BindingDB MonomerID,int64,0,0.000000,47957
BindingDB Ligand Name,BindingDB Ligand Name,str,0,0.000000,48085
Target Name,Target Name,str,0,0.000000,2034
Target Source Organism According to Curator or DataSource,Target Source Organism According to Curator or...,str,45,0.048019,297
Ki (nM),Ki (nM),object,67922,72.479512,5937
IC50 (nM),IC50 (nM),object,30187,32.212523,10853


##### **Data by PDSP**

In [11]:
data_pdsp = pd.read_csv(PDSP_DIR, sep='\t')
data_pdsp.info()

<class 'pandas.DataFrame'>
RangeIndex: 27715 entries, 0 to 27714
Columns: 640 entries, BindingDB Reactant_set_id to UniProt (TrEMBL) Alternative ID(s) of Target Chain 50
dtypes: float64(587), int64(6), str(47)
memory usage: 182.0 MB


C:\Users\alexl\AppData\Local\Temp\ipykernel_12564\117576025.py:1: DtypeWarning: Columns (0: UniProt (TrEMBL) Submitted Name of Target Chain 1, 1: UniProt (TrEMBL) Entry Name of Target Chain 1, 2: UniProt (TrEMBL) Primary ID of Target Chain 1, 3: UniProt (TrEMBL) Secondary ID(s) of Target Chain 1, 4: BindingDB Target Chain Sequence 2, 5: UniProt (SwissProt) Recommended Name of Target Chain 2, 6: UniProt (SwissProt) Entry Name of Target Chain 2, 7: UniProt (SwissProt) Primary ID of Target Chain 2, 8: UniProt (SwissProt) Secondary ID(s) of Target Chain 2, 9: BindingDB Target Chain Sequence 3, 10: UniProt (SwissProt) Recommended Name of Target Chain 3, 11: UniProt (SwissProt) Entry Name of Target Chain 3, 12: UniProt (SwissProt) Primary ID of Target Chain 3, 13: BindingDB Target Chain Sequence 4, 14: PDB ID(s) of Target Chain 4, 15: UniProt (SwissProt) Recommended Name of Target Chain 4, 16: UniProt (SwissProt) Entry Name of Target Chain 4, 17: UniProt (SwissProt) Primary ID of Target Chai

In [12]:
data_summry = pd.DataFrame({
    'Columns': data_pdsp.columns,
    'Data types': data_pdsp.dtypes,
    'Missing values': data_pdsp.isnull().sum(),
    'null%': data_pdsp.isnull().mean() * 100,
    'unique values': data_pdsp.nunique()
})

pd.set_option('display.max_rows', None)
display(data_summry)

,Columns,Data types,Missing values,null%,unique values
BindingDB Reactant_set_id,BindingDB Reactant_set_id,int64,0,0.000000,27715
Ligand SMILES,Ligand SMILES,str,0,0.000000,3303
Ligand InChI,Ligand InChI,str,0,0.000000,2990
Ligand InChI Key,Ligand InChI Key,str,0,0.000000,2990
BindingDB MonomerID,BindingDB MonomerID,int64,0,0.000000,3315
BindingDB Ligand Name,BindingDB Ligand Name,str,0,0.000000,3312
Target Name,Target Name,str,0,0.000000,400
Target Source Organism According to Curator or DataSource,Target Source Organism According to Curator or...,str,43,0.155151,41
Ki (nM),Ki (nM),str,0,0.000000,3972
IC50 (nM),IC50 (nM),float64,27715,100.000000,0
